In [ ]:
# File Loading and Preprocessing
import pandas as pd

games = pd.read_csv("data/steam_games_clean.csv")

print(games.shape)
print(games.head())


games["tags"] = games["tags"].fillna("")

games["tag_set"] = games["tags"].apply(
    lambda tags: {tag.strip() for tag in tags.split(",") if tag.strip()}
)

# Keep games with at least one tag
games = games[games["tag_set"].map(len) > 0].copy()

print(games[["name", "tag_set"]].head())

(23500, 10)
    app_id                                   name estimated_owners  price  \
0   496350  Supipara - Chapter 1 Spring Has Come!        0 - 20000   5.24   
1  1934300                     Armored Brigade II        0 - 20000  35.99   
2  1025440                     Fantasy General II  100000 - 200000  13.99   
3   328460                                Redline        0 - 20000   1.04   
4  2403000                 Fantasy Amusement Park        0 - 20000   5.99   

   positive  negative  avg_hours  median_hours               genres  \
0       252         3   0.133333      0.133333            Adventure   
1       117        13  11.250000     20.883333  Simulation,Strategy   
2      1025       210   8.150000      0.866667             Strategy   
3        74        35   3.383333      3.383333        Action,Racing   
4         0         0   1.466667      1.600000               Casual   

                                                tags  
0                  Adventure,Visual Novel,A

In [ ]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# convert sets of tags into a sparse binary matrix, 23000 x N_tags
mlb = MultiLabelBinarizer(sparse_output=True)
tag_matrix = mlb.fit_transform(games["tag_set"]).astype(np.float32)

# intersection of sparse matrices is analaogous to the dot product of the matrices
intersection = (tag_matrix @ tag_matrix.T).toarray()

# Compute the union of each pair of tag sets
tag_counts = np.asarray(tag_matrix.sum(axis=1)).ravel() 
counts_as_column = tag_counts.reshape(-1, 1)
counts_as_row = tag_counts.reshape(1, -1)

# Create the union array, then subtract intersections in place
union = counts_as_column + counts_as_row
union -= intersection

# Reuse the intersection array to store similarities
# empty pairs receive similarity 1
empty_pairs = union == 0
union[empty_pairs] = 1
intersection /= union
intersection[empty_pairs] = 1
similarity = intersection

# compute Jaccard similarity with vectorized division (way faster than looping through each pair of games)
similarity = np.divide(
    intersection, 
    union, 
    out=np.ones_like(intersection, dtype=np.float32), 
    where=union != 0
)

# Create a DataFrame for the similarity matrix with game names as both row and column labels
similarity_matrix = pd.DataFrame(
    similarity,
    index=games["name"],
    columns=games["name"],
)



In [ ]:
from sklearn.cluster import AgglomerativeClustering

distances = 1.0 - similarity_matrix.to_numpy()
np.fill_diagonal(distances, 0.0)

# Cluster games using average Jaccard distance between groups
model = AgglomerativeClustering(
    n_clusters=10,
    metric="precomputed",
    linkage="average",
)

# games must have the same row order as similarity_matrix
games = games.copy()
games["cluster"] = model.fit_predict(distances)


,game_count
cluster,
0,2
1,96
2,23
3,14
4,8
5,2
6,220
7,1
8,21162
